# 🎨 Indian Folk Art Restoration — Full Pipeline

**Models:** EDSR (super-resolution) + LaMa (inpainting GAN)  
**Art styles:** Madhubani · Warli · Pattachitra · Gond · Kalamkari  
**Hardware:** Google Colab T4 GPU (free tier)

---
Run cells top-to-bottom. The only manual step is uploading your `kaggle.json` API key in **Cell 3**.

In [ ]:
# ============================================================
# Cell 1 — Install all dependencies
# ============================================================
!pip install -q \
    torch torchvision \
    lpips \
    scikit-image \
    opencv-python-headless \
    kaggle \
    matplotlib \
    tqdm \
    Pillow \
    requests

print('✅ All dependencies installed.')

In [ ]:
# ============================================================
# Cell 2 — Mount Google Drive (for persistent checkpoint saves)
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/FolkArtRestoration'
os.makedirs(DRIVE_ROOT, exist_ok=True)
print(f'Drive root: {DRIVE_ROOT}')

In [ ]:
# ============================================================
# Cell 3 — Upload kaggle.json and configure Kaggle API
# ============================================================
from google.colab import files

print('Upload your kaggle.json API key:')
uploaded = files.upload()   # select kaggle.json from your computer

import os, shutil
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
shutil.move('kaggle.json', os.path.expanduser('~/.kaggle/kaggle.json'))
os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)
print('✅ Kaggle credentials configured.')

In [ ]:
# ============================================================
# Cell 4 — Download & extract dataset
# ============================================================
import subprocess, zipfile, os, requests
from pathlib import Path

RAW_DIR = Path('/content/folk_art_restoration/data/raw')
RAW_DIR.mkdir(parents=True, exist_ok=True)

def download_kaggle_dataset(dataset_slug: str, dest: Path) -> bool:
    """Try downloading a Kaggle dataset; return True on success."""
    try:
        result = subprocess.run(
            ['kaggle', 'datasets', 'download', '-d', dataset_slug, '--unzip', '-p', str(dest)],
            capture_output=True, text=True, timeout=300
        )
        if result.returncode == 0:
            print(f'  ✅ Downloaded: {dataset_slug}')
            return True
        else:
            print(f'  ⚠ Failed ({dataset_slug}): {result.stderr[:200]}')
            return False
    except Exception as e:
        print(f'  ✗ Exception: {e}')
        return False


def scrape_wikimedia(class_name: str, dest: Path, n: int = 50) -> int:
    """
    Fallback: fetch images from Wikimedia Commons API for a given folk-art style.
    Returns number of images downloaded.
    """
    dest.mkdir(parents=True, exist_ok=True)
    query = class_name.replace('_', ' ') + ' painting'
    url   = 'https://commons.wikimedia.org/w/api.php'
    params = {
        'action': 'query', 'list': 'search', 'srsearch': query,
        'srnamespace': '6',  # File namespace
        'srlimit': min(n * 2, 100), 'format': 'json'
    }
    try:
        resp = requests.get(url, params=params, timeout=15)
        data = resp.json()
    except Exception as e:
        print(f'  Wikimedia search failed: {e}')
        return 0

    count = 0
    for item in data.get('query', {}).get('search', []):
        title = item['title']       # e.g. "File:Madhubani.jpg"
        if not any(title.lower().endswith(ext) for ext in ['.jpg', '.jpeg', '.png']):
            continue
        # Get image URL
        info_params = {
            'action': 'query', 'titles': title,
            'prop': 'imageinfo', 'iiprop': 'url',
            'format': 'json'
        }
        try:
            info = requests.get(url, params=info_params, timeout=10).json()
            pages = info['query']['pages']
            img_url = next(iter(pages.values()))['imageinfo'][0]['url']
            # Download image
            img_data = requests.get(img_url, timeout=20).content
            fname = dest / f'{class_name}_{count:04d}{Path(img_url).suffix}'
            fname.write_bytes(img_data)
            count += 1
            if count >= n:
                break
        except Exception:
            continue

    print(f'  Wikimedia {class_name}: {count} images downloaded')
    return count


# ----- Primary dataset attempt -------------------------------------------
PRIMARY   = 'mcmillanajhonda/indian-folk-art'
FALLBACK1 = 'vikasukani/indian-paintings-dataset'
FALLBACK2 = 'metabrainz/wiki-art'   # large; filtered below

success = download_kaggle_dataset(PRIMARY, RAW_DIR)

if not success:
    print('Primary dataset unavailable; trying fallback …')
    success = download_kaggle_dataset(FALLBACK1, RAW_DIR)

# ----- Supplement with Wikimedia if any class is under-represented -------
CLASSES   = ['Madhubani', 'Warli', 'Pattachitra', 'Gond', 'Kalamkari']
MIN_COUNT = 200

print('\nChecking class counts …')
for cls in CLASSES:
    class_dir = RAW_DIR / cls
    class_dir.mkdir(exist_ok=True)
    # Count images already present (case-insensitive folder search)
    found = []
    for p in RAW_DIR.rglob('*'):
        if p.is_file() and cls.lower() in str(p).lower() \
                and p.suffix.lower() in {'.jpg', '.jpeg', '.png'}:
            found.append(p)

    print(f'  {cls}: {len(found)} images found', end='')
    if len(found) < MIN_COUNT:
        need = MIN_COUNT - len(found)
        print(f' → scraping {need} more from Wikimedia …')
        scrape_wikimedia(cls, class_dir, n=need)
    else:
        print(' ✓')

print('\n✅ Dataset ready.')
# Quick sanity: count total images
all_imgs = list(RAW_DIR.rglob('*.jpg')) + list(RAW_DIR.rglob('*.png')) + list(RAW_DIR.rglob('*.jpeg'))
print(f'Total images in raw/: {len(all_imgs)}')

In [ ]:
# ============================================================
# Cell 5 — Clone / write project files
# ============================================================
# If you cloned the repo from GitHub, just run:
#   !git clone https://github.com/<you>/folk-art-restoration /content/folk_art_restoration
#   %cd /content/folk_art_restoration
#
# Otherwise, the project files are assumed already at /content/folk_art_restoration/

import sys, os
PROJECT_ROOT = '/content/folk_art_restoration'
os.makedirs(PROJECT_ROOT, exist_ok=True)
os.chdir(PROJECT_ROOT)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Create __init__ files so Python treats dirs as packages
for pkg in ['models', 'utils']:
    init = os.path.join(PROJECT_ROOT, pkg, '__init__.py')
    os.makedirs(os.path.dirname(init), exist_ok=True)
    open(init, 'a').close()

print(f'Working directory: {os.getcwd()}')
print('Python path updated.')

In [ ]:
# ============================================================
# Cell 6 — Create damaged dataset (clean + damaged + masks + splits)
# ============================================================
from utils.degrade import create_damaged_dataset

INPUT_DIR  = '/content/folk_art_restoration/data/raw'
OUTPUT_DIR = '/content/folk_art_restoration/data'

create_damaged_dataset(
    input_dir=INPUT_DIR,
    output_dir=OUTPUT_DIR,
    mode='random',        # random damage type per image
    train_ratio=0.70,
    val_ratio=0.15,
    seed=42,
)

# Quick visual sanity-check: show one damaged example
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

sample_files = sorted(Path(OUTPUT_DIR, 'clean', 'train').glob('*.png'))[:1]
if sample_files:
    fname = sample_files[0].name
    fig, axes = plt.subplots(1, 3, figsize=(11, 4))
    for ax, subdir, title in zip(axes,
                                  ['clean', 'damaged', 'masks'],
                                  ['Clean', 'Damaged', 'Mask']):
        img = Image.open(Path(OUTPUT_DIR, subdir, 'train', fname))
        ax.imshow(img, cmap='gray' if subdir == 'masks' else None)
        ax.set_title(title); ax.axis('off')
    plt.suptitle(f'Sample: {fname}', fontsize=12)
    plt.tight_layout()
    plt.show()

print('\n✅ Damaged dataset created.')

In [ ]:
# ============================================================
# Cell 7 — Train EDSR (~2 hours on T4)
# ============================================================
import subprocess

edsr_ckpt_dir = f'{DRIVE_ROOT}/checkpoints/edsr'

result = subprocess.run(
    [
        'python', 'train_edsr.py',
        '--data_root',   '/content/folk_art_restoration/data',
        '--ckpt_dir',    edsr_ckpt_dir,
        '--scale',       '2',
        '--epochs',      '50',
        '--batch_size',  '16',
        '--num_workers', '2',
    ],
    cwd='/content/folk_art_restoration',
    # Stream output to notebook:
    stdout=None, stderr=None
)
print(f'\nEDSR training exit code: {result.returncode}')

In [ ]:
# ============================================================
# Cell 8 — Plot EDSR training curves
# ============================================================
from utils.visualize import plot_training_curves
import os

edsr_log = f'{DRIVE_ROOT}/checkpoints/edsr/train_log.csv'
if os.path.exists(edsr_log):
    plot_training_curves(edsr_log, title='EDSR Training Curves')
    from IPython.display import Image as IPImage
    display(IPImage(edsr_log.replace('.csv', '.png')))
else:
    print('Log not found — has training completed?')

In [ ]:
# ============================================================
# Cell 9 — Train LaMa GAN (~4 hours on T4)
# ============================================================
import subprocess

lama_ckpt_dir = f'{DRIVE_ROOT}/checkpoints/lama'

result = subprocess.run(
    [
        'python', 'train_lama.py',
        '--data_root',   '/content/folk_art_restoration/data',
        '--ckpt_dir',    lama_ckpt_dir,
        '--epochs',      '100',
        '--batch_size',  '8',
        '--num_workers', '2',
    ],
    cwd='/content/folk_art_restoration',
    stdout=None, stderr=None
)
print(f'\nLaMa training exit code: {result.returncode}')

In [ ]:
# ============================================================
# Cell 10 — Plot LaMa training curves
# ============================================================
from utils.visualize import plot_training_curves
import os

lama_log = f'{DRIVE_ROOT}/checkpoints/lama/train_log.csv'
if os.path.exists(lama_log):
    plot_training_curves(lama_log, title='LaMa Training Curves')
    from IPython.display import Image as IPImage
    display(IPImage(lama_log.replace('.csv', '.png')))
else:
    print('Log not found.')

In [ ]:
# ============================================================
# Cell 11 — Full evaluation & comparison table
# ============================================================
import subprocess

result = subprocess.run(
    [
        'python', 'evaluate.py',
        '--data_root',  '/content/folk_art_restoration/data',
        '--edsr_ckpt',  f'{DRIVE_ROOT}/checkpoints/edsr/edsr_best.pth',
        '--lama_ckpt',  f'{DRIVE_ROOT}/checkpoints/lama/lama_best.pth',
        '--output_dir', '/content/folk_art_restoration/data/restored',
        '--scale',      '2',
        '--n_vis',      '5',
    ],
    cwd='/content/folk_art_restoration',
    stdout=None, stderr=None
)
print(f'Evaluation exit code: {result.returncode}')

In [ ]:
# ============================================================
# Cell 12 — Display 5 before/after restoration examples
# ============================================================
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path

vis_dir = Path('/content/folk_art_restoration/data/restored/visualisations')
vis_files = sorted(vis_dir.glob('*.png'))[:5]

if vis_files:
    for vf in vis_files:
        img = mpimg.imread(str(vf))
        fig, ax = plt.subplots(figsize=(13, 4))
        ax.imshow(img)
        ax.axis('off')
        ax.set_title(vf.stem, fontsize=10)
        plt.tight_layout()
        plt.show()
else:
    print('No visualisation files found. Has evaluate.py been run?')

In [ ]:
# ============================================================
# Cell 13 — Display final metrics comparison table
# ============================================================
import json
from pathlib import Path

results_json = Path('/content/folk_art_restoration/data/restored/eval_results.json')
if results_json.exists():
    data = json.loads(results_json.read_text())
    print(f'{'Model':<26} | {"PSNR ↑":>10} | {"SSIM ↑":>8} | {"LPIPS ↓":>8}')
    print('-' * 60)
    for model_name, metrics in data.items():
        psnr  = f"{metrics['psnr_mean']:.2f}"
        ssim  = f"{metrics['ssim_mean']:.3f}"
        lpips = f"{metrics['lpips_mean']:.3f}" if metrics['lpips_mean'] is not None else 'N/A'
        print(f'{model_name:<26} | {psnr:>10} | {ssim:>8} | {lpips:>8}')
else:
    print('Results JSON not found.')

In [ ]:
# ============================================================
# Cell 14 — Single-image demo (restore one image interactively)
# ============================================================
from google.colab import files
import subprocess
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

print('Upload a damaged folk-art image to restore:')
uploaded = files.upload()

for filename in uploaded:
    input_path  = f'/content/{filename}'
    output_path = f'/content/restored_{filename}'

    result = subprocess.run(
        [
            'python', 'restore.py',
            '--input',     input_path,
            '--output',    output_path,
            '--edsr_ckpt', f'{DRIVE_ROOT}/checkpoints/edsr/edsr_best.pth',
            '--lama_ckpt', f'{DRIVE_ROOT}/checkpoints/lama/lama_best.pth',
        ],
        cwd='/content/folk_art_restoration',
        capture_output=True, text=True
    )
    print(result.stdout)
    if result.returncode == 0 and Path(output_path).exists():
        fig, axes = plt.subplots(1, 2, figsize=(10, 5))
        axes[0].imshow(mpimg.imread(input_path));  axes[0].set_title('Damaged Input'); axes[0].axis('off')
        axes[1].imshow(mpimg.imread(output_path)); axes[1].set_title('Restored');      axes[1].axis('off')
        plt.suptitle(filename, fontsize=12)
        plt.tight_layout()
        plt.show()
        # Offer download
        files.download(output_path)
    else:
        print('Restoration failed:', result.stderr[:500])